# Intent Classification Module

This notebook covers the training, evaluation, and saving of an Intent Classifier for the RAG chatbot.

## 1. Load Dataset
We use the `bitext/Bitext-customer-support-llm-chatbot-training-dataset` from Hugging Face.

In [ ]:
# Disable tqdm progress bars to prevent ipykernel ContextVar errors
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import datasets
datasets.disable_progress_bar()

from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = dataset.to_pandas()
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(y='intent', data=df, order=df['intent'].value_counts().index)
plt.title('Distribution of 27 Fine-grained Intents')
plt.show()

plt.figure(figsize=(8,5))
sns.countplot(y='category', data=df, order=df['category'].value_counts().index)
plt.title('Distribution of 10 Categories')
plt.show()

## 3. Map Intents to Condensed Groups
We map the 27 intents to 7 broader categories to simplify the classification task for the RAG system.

In [ ]:
import sys
sys.path.append('..')
from src.config import INTENT_MAP

df['condensed_intent'] = df['intent'].map(INTENT_MAP)

plt.figure(figsize=(8,4))
sns.countplot(y='condensed_intent', data=df, order=df['condensed_intent'].value_counts().index)
plt.title('Distribution of Condensed Intents')
plt.show()

## 4. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

df = df.dropna(subset=['condensed_intent'])
X = df['instruction']
y = df['condensed_intent']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=1/9, stratify=y_temp, random_state=42)

print(f'Train size: {len(X_train)}\nVal size: {len(X_val)}\nTest size: {len(X_test)}')

## 5. Preprocessing

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(cleaned_tokens)

X_train_clean = X_train.apply(preprocess_text)
X_val_clean = X_val.apply(preprocess_text)
X_test_clean = X_test.apply(preprocess_text)

## 6. Feature Extraction (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50000)
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_val_tfidf = vectorizer.transform(X_val_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

## 7. Model Training & Comparison
Why traditional ML? Intent classification on well-structured customer support queries (like those generated in this bitext dataset) is highly reliant on key phrases and vocabulary. Models like LinearSVC or Logistic Regression with TF-IDF features are incredibly fast, lightweight, and often achieve >95% accuracy for these straightforward mappings, making LLMs or deep learning overkill for this specific routing step.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    'LinearSVC': LinearSVC(random_state=42),
    'Logistic Regression': LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_val_tfidf)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average='macro')
    print(f"{name} - Val Accuracy: {acc:.4f}, Val Macro F1: {f1:.4f}")

## 8. Hyperparameter Tuning (LinearSVC)
LinearSVC usually performs best or very close to best, while being extremely fast.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.1, 1, 10]}
grid = GridSearchCV(LinearSVC(random_state=42), param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train_tfidf, y_train)

best_model = grid.best_estimator_
print(f"Best Parameters: {grid.best_params_}")

## 9. Evaluation on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_test_pred = best_model.predict(X_test_tfidf)
print("Test Set Evaluation:")
print(classification_report(y_test, y_test_pred))

plt.figure(figsize=(8,6))
cm = confusion_matrix(y_test, y_test_pred, labels=best_model.classes_)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=best_model.classes_, yticklabels=best_model.classes_, cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 10. Save Model and Vectorizer

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
from src.config import INTENT_MODEL_PATH, INTENT_VECTORIZER_PATH

joblib.dump(best_model, INTENT_MODEL_PATH)
joblib.dump(vectorizer, INTENT_VECTORIZER_PATH)
print("Model and vectorizer saved successfully!")